In [1]:
#Step 1 — Parse and clean input data
#Loops over results, extracts candidate metadata (cand_dict) and folding output (plot_info), normalizes formats 
#(handles list/None cases), and prepares required fields (notes, file paths, etc.).

#Step 2 — Compute and organize candidate parameters
#Builds derived quantities: converts RA/Dec using SkyCoord, selects best values for frequency (f0), dispersion measure (dm),
#SNR, and constructs paths to diagnostic plots and summary PDFs.

#Step 3 — Upload candidate to database
#Calls sd.add_candidate(...) to register each candidate with all computed parameters and file links, 
#then prints verification info (with optional final commit to save changes).

import click
from sps_pipeline.candidate_viewer import CandidateViewerRegistrar
import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import scipy
import numpy as np
from datetime import datetime, timedelta
from sps_databases import db_utils, db_api
from pathlib import Path
import astropy.units as u
from astropy.coordinates import SkyCoord

In [13]:
# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',#no password for automation user
    'database': 'champss'
}
db = db_utils.connect(name="champss_processing")

#We will create a separate folder in the multiday folds survey, called followup, or something along those lines.
#Maybe we create one new folder for the followup sources from each month.

sd = CandidateViewerRegistrar(
        survey="multiday",  #the project name under top-right corner of the website
        folder="single_day_3_pdf", #create a new folder
        db_config=db_config,
        survey_dir="/data/candidate_viewer/champss_candidate_viewer/surveys",  # path to the directory containing survey (project) config files
    )

In [14]:
# Load results
results = np.load("result_single_day31.pkl", allow_pickle=True)
print(results)

[[{'file': 'Multi_Pointing_Groups_f_15.130_DM_54.546_69c11f93ff2c065afeea4e0e', 'folder': '2026-03-22', 'result': '<faint>', 'date': 1774377283, 'modified_by': 'Viewer Bot', 'info': '{"history":[]}', 'id': 8948, 'rater_a': 'Lars Kuenkel', 'result_a': '<faint>', 'rater_b': 'Vicky Kaspi', 'result_b': '<faint>', 'additional_ratings': '{}', 'metadata': {'survey': 'dailycands', 'folder': '2026-03-22', 'file': 'Multi_Pointing_Groups_f_15.130_DM_54.546_69c11f93ff2c065afeea4e0e', 'input_file': '/mnt/beegfs-client/processed/mp_runs/daily_20260322/candidates/Multi_Pointing_Groups_f_15.130_DM_54.546_69c11f93ff2c065afeea4e0e.npz', 'candidate': '', 'telescope': 'chime', 'epoch_topo': '', 'epoch_bary': '', 't_sample': '', 'data_folded': '', 'data_avg': '', 'data_stdev': '', 'profile_bins': '', 'profile_avg': '', 'profile_stdev': '', 'reduce_chi_sqr': '', 'prob_noise': '7.785725116729736', 'best_dm': '54.54568594474638', 'p_topo': '', 'p_topo_d1': '', 'p_topo_d2': '', 'p_bary': '66.09336695562965', '

In [15]:
#Adding candidate to website
counter = 0
for i, entry in enumerate(results):
    counter += 1
    # Get the input file from metadata
    meta_entry, plot_entry = entry
    md = meta_entry.get("metadata", {})
    file = md.get("input_file")
    
    if not file:
        print(f"Entry {i} has no input_file, skipping.") # to avoid error
        continue
    
    # Query the database for this file with the extra filter
    db_entry = db.followup_sources.find_one({
        'source_type': 'md_candidate',
        'path_to_candidates': file
    })
    print(db_entry["source_name"])
    # Get the pdf path
    if not db_entry:
        print(f"No database entry found for file: {file} with source_type='md_candidate'") # to avoid error
        continue

    path = db_entry.get("coherentsearch_history", [])
    if isinstance(path, list) and len(path) > 0:
        plot_info = path[0]   # use first history entry
    else:
        plot_info = {}

    if not path:
        print(f"Entry {i} has no history")
        continue

    mdf_plot = path[0]["path_to_plot"]
    mdf_plot_path = Path(mdf_plot)
    phase_plot = mdf_plot
    
    # Extract pdf format path
    source_name = db_entry["source_name"]
    source_name = source_name.replace("/", "_").replace(" ", "_") #using the code from git repo in multidayfold/summary_plot

    summary_pdf = mdf_plot_path.parent / f"{source_name}_summary.pdf"


    # Getting the parameter
    # Convert RA/Dec
    coord = SkyCoord(
        md.get('source_ra', '0:0:0'),
        md.get('source_dec', '0:0:0'),
        unit=(u.hourangle, u.deg)
    )

    ra_deg = coord.ra.deg
    dec_deg = coord.dec.deg

    notes = md.get("notes", {})

    file = md.get("input_file")
    if not file:
        continue

    db_entry = db.followup_sources.find_one({
        'source_type': 'md_candidate',
        'path_to_candidates': file
    })

    if not db_entry:
        continue

    f0_val = float(plot_info.get('f0', md.get('freq', 0)))
    dm_val = float(md.get('best_dm', 0))
    snr_val = float(plot_info.get('SN', notes.get('fs_sigma', 0)))

    input_file_val = md.get('input_file', '')
    fs_id_val = notes.get('fs_id', 'not_specified')
    fs_sigma_val = notes.get('fs_sigma', 'not_specified')
    fs_file_val = notes.get('fs_file', 'not_specified')

    sd.add_candidate(
        candname=md.get('file', 'unknown'),
        ra=ra_deg,
        dec=dec_deg,
        f0=f0_val,
        dm=dm_val,
        snr=snr_val,
        input_file=input_file_val,
        fs_id=fs_id_val,
        fs_sigma=fs_sigma_val,
        fs_file=fs_file_val,
        phase_search_diagnostics=phase_plot,
        summary_pdf=summary_pdf,
        #**{k: v for k, v in md.items() if k not in [
            #'file','source_ra','source_dec','freq','best_dm','input_file','notes'
        #]}
    )

    # Debug print
    print(f"Candidate #{counter}")
    print("RA:", ra_deg)
    print("Dec:", dec_deg)
    print("DM:", dm_val)
    print("f0:", f0_val)
    print("SNR:", snr_val)
    print("Phase plot:", phase_plot)
    print("PDF path:", summary_pdf)
    print("-" * 40)

#sd.commit()  # commit!!! to database and update survey config ,after the loop line 310 git
#Phase plot: /mnt/beegfs-client/processed/archives//candidates/227.88_55.61//phase_search_22.26_1.36.png
#PDF path: /mnt/beegfs-client/processed/archives/candidates/227.88_55.61/md_227.88_55.61_1.364242_22.26_summary.pdf
#multiday search, summary plot in git. use source name instead of my forced one
#glob fct to find file look at theory(in bash or python)

md_211.31_72.11_15.130111_54.55
Candidate #1
RA: 211.3119147602083
Dec: 72.10505911758055
DM: 54.54568594474638
f0: 15.13011836333806
SNR: 6.110917091369629
Phase plot: /mnt/beegfs-client/processed/archives//candidates/211.31_72.11//phase_search_54.55_15.13.png
PDF path: /mnt/beegfs-client/processed/archives/candidates/211.31_72.11/md_211.31_72.11_15.130111_54.55_summary.pdf
----------------------------------------
md_182.51_63.14_25.367505_26.62
Candidate #2
RA: 182.51256662329163
Dec: 63.13595805909722
DM: 26.61505640717681
f0: 25.36751455120496
SNR: 4.303627967834473
Phase plot: /mnt/beegfs-client/processed/archives//candidates/182.51_63.14//phase_search_26.62_25.37.png
PDF path: /mnt/beegfs-client/processed/archives/candidates/182.51_63.14/md_182.51_63.14_25.367505_26.62_summary.pdf
----------------------------------------
md_145.17_69.6_3.078988_121.94
Candidate #3
RA: 145.168248216375
Dec: 69.60247010670555
DM: 121.94350939409908
f0: 3.0789886147521557
SNR: 7.561635494232178
Phas

Registering candidates: 100%|███████████████████████████████████████████| 37/37 [00:00<00:00, 655.63it/s]
